In [1]:
# 高性能VAE缺失值填补模型 - 最终优化版本（移除HistGradientBoosting）
import tensorflow as tf
import numpy as np
import pandas as pd
from tensorflow import keras
from tensorflow.keras import layers, optimizers, regularizers, initializers, callbacks
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, QuantileTransformer, PowerTransformer
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from scipy import stats
from scipy import special
import warnings
import random
import os
import gc
import time
import hashlib

warnings.filterwarnings("ignore")

# 检查TensorFlow版本
print(f"TensorFlow版本: {tf.__version__}")

# ==================== 强化的可重复性设置 ====================
SEED = 42

def set_deterministic_mode():
    """设置完全确定性模式"""
    # Python随机种子
    random.seed(SEED)
    
    # Python哈希种子
    os.environ['PYTHONHASHSEED'] = str(SEED)
    
    # Numpy随机种子
    np.random.seed(SEED)
    
    # TensorFlow随机种子
    tf.random.set_seed(SEED)
    
    # TensorFlow确定性设置
    os.environ['TF_DETERMINISTIC_OPS'] = '1'
    os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
    
    # 禁用TensorFlow的多线程（确保确定性）
    tf.config.threading.set_inter_op_parallelism_threads(1)
    tf.config.threading.set_intra_op_parallelism_threads(1)
    
    print("已启用完全确定性模式")

# 初始化确定性模式
set_deterministic_mode()

# GPU设置（使用确定性模式或禁用GPU）
USE_GPU = False  # 设为False以确保完全可重复（GPU操作可能有非确定性）

if USE_GPU:
    gpus = tf.config.experimental.list_physical_devices('GPU')
    if gpus:
        try:
            # 限制GPU内存增长
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
            # 设置确定性GPU操作
            tf.config.experimental.enable_op_determinism()
            print(f"发现 {len(gpus)} 个GPU设备，已启用确定性模式")
        except RuntimeError as e:
            print(f"GPU设置错误: {e}")
    else:
        print("未发现GPU，使用CPU运行")
else:
    # 禁用GPU
    os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
    print("已禁用GPU，使用CPU确保完全可重复性")

# 清理会话
tf.keras.backend.clear_session()
gc.collect()

# 创建随机数生成器（用于需要随机数的地方）
rng = np.random.RandomState(SEED)
tf_random = tf.random.Generator.from_seed(SEED)

# 手动实现反Box-Cox变换
def inv_boxcox(y, lambda_param):
    """手动实现反Box-Cox变换"""
    if lambda_param == 0:
        return np.exp(y)
    else:
        # 避免负值导致的问题
        return np.power(lambda_param * y + 1, 1 / lambda_param)

# 1. 数据加载和预处理
def load_and_preprocess_data():
    """加载数据并进行初步预处理"""
    # 使用固定的随机种子
    local_rng = np.random.RandomState(SEED)
    
    data = pd.read_csv("./Rainfall.csv", low_memory=True)
    print(f"原始数据形状: {data.shape}")
    
    # 保存原始数据副本
    original_data = data.copy()
    
    # 删除非数值列（确定性处理）
    cols_to_drop = []
    for col in sorted(data.columns):  # 排序确保顺序一致
        try:
            pd.to_numeric(data[col].dropna().iloc[:10])
        except:
            cols_to_drop.append(col)
    
    data = data.drop(columns=sorted(cols_to_drop))  # 排序确保顺序一致
    
    # 删除ID列（确定性处理）
    cols_sorted = sorted(data.columns[:3])  # 排序确保顺序一致
    for col in cols_sorted:
        if data[col].dtype in ['int64', 'float64']:
            non_null = data[col].dropna()
            if len(non_null) > 1:
                diffs = non_null.diff().dropna()
                if (diffs == 1).mean() > 0.9:
                    data = data.drop(columns=[col])
    
    # 删除L(m)列
    if 'L(m)' in data.columns:
        data = data.drop(columns=['L(m)'])
    
    # 移除缺失率过高的列（>30%）
    missing_ratio = data.isnull().sum() / len(data)
    cols_to_drop = sorted(missing_ratio[missing_ratio > 0.30].index)  # 排序确保一致
    data = data.drop(columns=cols_to_drop)
    
    # 转换数据类型
    for col in sorted(data.columns):  # 排序确保顺序一致
        data[col] = pd.to_numeric(data[col], errors='coerce').astype('float32')
    
    # 重新排序列（确保顺序一致）
    data = data[sorted(data.columns)]
    
    print(f"最终数据形状: {data.shape}")
    print(f"特征: {list(data.columns)}")
    
    return data, original_data

# 2. 优化的异常值处理（As使用更温和的处理）
def optimized_outlier_handling(df):
    """优化的异常值处理，平衡各特征的处理强度"""
    df_clean = df.copy()
    transform_info = {}
    
    # 排序列名确保处理顺序一致
    columns_sorted = sorted(df_clean.columns)
    
    for column in columns_sorted:
        mask = df_clean[column].notna()
        values = df_clean.loc[mask, column].values
        
        if len(values) == 0:
            continue
        
        # V(m³)特殊处理 - 保持原有处理方式
        if 'V' in column:
            print(f"{column}: 使用Quantile变换处理极端分布")
            
            qt = QuantileTransformer(n_quantiles=min(1000, len(values)), 
                                    output_distribution='normal', 
                                    random_state=SEED,
                                    subsample=100000)
            values_transformed = qt.fit_transform(values.reshape(-1, 1)).flatten()
            
            transform_info[column] = {'type': 'quantile', 'transformer': qt}
            df_clean.loc[mask, column] = values_transformed
            
        elif 'H' in column:
            # H(m)保持简化处理（已经表现良好）
            print(f"{column}: 使用简化的稳健处理（保留原始信息）")
            
            # 只进行温和的异常值裁剪
            q1, q3 = np.percentile(values, [10, 90])
            iqr = q3 - q1
            lower = q1 - 2.0 * iqr
            upper = q3 + 2.0 * iqr
            values_clipped = np.clip(values, lower, upper)
            
            # 简单的稳健标准化
            median = np.median(values_clipped)
            mad = np.median(np.abs(values_clipped - median))
            if mad > 0:
                values_standardized = (values_clipped - median) / (mad * 1.4826)
            else:
                values_standardized = values_clipped - median
            
            transform_info[column] = {
                'type': 'h_robust', 
                'clip_lower': lower,
                'clip_upper': upper,
                'median': median,
                'mad': mad
            }
            df_clean.loc[mask, column] = values_standardized
            
        elif 'As' in column:
            # As改进处理 - 使用更温和的方法保留更多信息
            print(f"{column}: 使用温和的PowerTransformer（保留信息）")
            
            # 更温和的异常值处理（扩大范围）
            q1, q3 = np.percentile(values, [5, 95])
            iqr = q3 - q1
            lower = q1 - 2.5 * iqr  # 放宽裁剪
            upper = q3 + 2.5 * iqr
            values = np.clip(values, lower, upper)
            
            # 使用PowerTransformer代替Box-Cox（更稳定）
            pt = PowerTransformer(method='yeo-johnson', standardize=True, copy=True)
            values_transformed = pt.fit_transform(values.reshape(-1, 1)).flatten()
            
            transform_info[column] = {
                'type': 'power', 
                'transformer': pt,
                'clip_lower': lower,
                'clip_upper': upper
            }
            df_clean.loc[mask, column] = values_transformed
            
        elif 'θ' in column:
            # 角度特征：继续增强正则化处理
            print(f"{column}: 增强正则化处理")
            
            # 激进的异常值裁剪
            lower, upper = np.percentile(values, [3, 97])
            values = np.clip(values, lower, upper)
            
            # 添加噪声防止过拟合
            noise = np.random.RandomState(SEED).normal(0, 0.025 * np.std(values), len(values))
            values = values + noise
            
            # 稳健标准化
            median = np.median(values)
            mad = np.median(np.abs(values - median))
            if mad > 0:
                values = (values - median) / (mad * 1.4826)
            else:
                values = values - median
            
            transform_info[column] = {
                'type': 'theta_regularized', 
                'median': median,
                'mad': mad,
                'lower': lower,
                'upper': upper
            }
            df_clean.loc[mask, column] = values
            
        else:
            # Wsm(m)：保持稳健处理
            print(f"{column}: 稳健异常值处理")
            
            # 使用1.5倍IQR
            q1, q3 = np.percentile(values, [25, 75])
            iqr = q3 - q1
            lower = q1 - 1.5 * iqr
            upper = q3 + 1.5 * iqr
            values = np.clip(values, lower, upper)
            transform_info[column] = {'type': 'clip', 'shift': 0}
            df_clean.loc[mask, column] = values
    
    return df_clean, transform_info

# 3. 优化的标准化（As使用更适合的方法）
def optimized_normalization(data):
    """优化的标准化方法"""
    data_normalized = pd.DataFrame(index=data.index, columns=data.columns)
    scalers = {}
    
    # 排序列名确保处理顺序一致
    columns_sorted = sorted(data.columns)
    
    for column in columns_sorted:
        mask = data[column].notna()
        values = data[column][mask].values.reshape(-1, 1)
        
        if len(values) > 0:
            # 根据特征选择不同的标准化方法
            if 'V' in column:
                # V使用StandardScaler
                scaler = StandardScaler()
            elif 'H' in column:
                # H(m)保持RobustScaler（已经表现良好）
                scaler = RobustScaler(quantile_range=(10, 90))
            elif 'As' in column:
                # As使用StandardScaler以保留更多信息
                scaler = StandardScaler()
            elif 'θ' in column:
                # θ使用严格的RobustScaler（增强正则化）
                scaler = RobustScaler(quantile_range=(25, 75))
            else:
                # 其他使用RobustScaler
                scaler = RobustScaler(quantile_range=(5, 95))
            
            scaler.fit(values)
            scaled = scaler.transform(values).flatten()
            
            # 映射到合适范围
            min_val, max_val = scaled.min(), scaled.max()
            if max_val - min_val > 1e-8:
                if 'H' in column:
                    # H(m)保持宽范围
                    scaled = (scaled - min_val) / (max_val - min_val) * 0.9 + 0.05
                elif 'As' in column:
                    # As使用全范围以保留信息
                    scaled = (scaled - min_val) / (max_val - min_val) * 0.95 + 0.025
                elif 'θ' in column:
                    # θ使用窄范围（正则化）
                    scaled = (scaled - min_val) / (max_val - min_val) * 0.5 + 0.25
                else:
                    scaled = (scaled - min_val) / (max_val - min_val) * 0.8 + 0.1
            else:
                scaled = np.ones_like(scaled) * 0.5
            
            data_normalized.loc[mask, column] = scaled.astype(np.float32)
            scalers[column] = {'scaler': scaler, 'min': min_val, 'max': max_val}
    
    return data_normalized, scalers

# 修复的反标准化函数
def inverse_transform_array(data_normalized_array, scalers, transform_info, original_columns, original_index=None):
    """将标准化的numpy数组转换回原始尺度的DataFrame"""
    n_samples = data_normalized_array.shape[0]
    
    if original_index is None:
        original_index = range(n_samples)
    
    data_original = pd.DataFrame(index=original_index, columns=original_columns)
    
    for i, column in enumerate(original_columns):
        if column not in scalers:
            data_original[column] = data_normalized_array[:, i]
            continue
            
        values = data_normalized_array[:, i].copy()
        
        # 反标准化
        scaler_info = scalers[column]
        min_val = scaler_info['min']
        max_val = scaler_info['max']
        
        # 从映射范围恢复
        if 'H' in column:
            values = (values - 0.05) / 0.9 * (max_val - min_val) + min_val
        elif 'As' in column:
            values = (values - 0.025) / 0.95 * (max_val - min_val) + min_val
        elif 'θ' in column:
            values = (values - 0.25) / 0.5 * (max_val - min_val) + min_val
        else:
            values = (values - 0.1) / 0.8 * (max_val - min_val) + min_val
        
        # 反Scaler
        values = scaler_info['scaler'].inverse_transform(values.reshape(-1, 1)).flatten()
        
        # 反变换
        if column in transform_info:
            trans_type = transform_info[column]['type']
            
            if trans_type == 'quantile':
                # 反Quantile变换
                transformer = transform_info[column]['transformer']
                values = transformer.inverse_transform(values.reshape(-1, 1)).flatten()
            elif trans_type == 'h_robust':
                # H(m)的简化反变换
                median = transform_info[column]['median']
                mad = transform_info[column]['mad']
                if mad > 0:
                    values = values * (mad * 1.4826) + median
                else:
                    values = values + median
            elif trans_type == 'power':
                # As的PowerTransformer反变换
                transformer = transform_info[column]['transformer']
                values = transformer.inverse_transform(values.reshape(-1, 1)).flatten()
            elif trans_type == 'theta_regularized':
                # θ的反变换
                median = transform_info[column]['median']
                mad = transform_info[column]['mad']
                if mad > 0:
                    values = values * (mad * 1.4826) + median
                else:
                    values = values + median
        
        data_original[column] = values
    
    return data_original

# 4. 优化的VAE模型（平衡各特征权重）
class OptimizedVAE:
    def __init__(self, data_dim, latent_dim=14, learning_rate=0.0004, seed=42, column_names=None):
        """优化的VAE模型"""
        self.data_dim = data_dim
        self.latent_dim = latent_dim
        self.learning_rate = learning_rate
        self.seed = seed
        self.column_names = column_names
        
        # 创建特征权重掩码
        self.feature_weights = np.ones(data_dim, dtype=np.float32)
        self.regularization_weights = np.ones(data_dim, dtype=np.float32)
        
        if column_names is not None:
            for i, col in enumerate(column_names):
                if 'H' in col:
                    self.feature_weights[i] = 4.0  # 保持H(m)的高权重
                    self.regularization_weights[i] = 0.4
                elif 'As' in col:
                    self.feature_weights[i] = 3.0  # 增强As的权重
                    self.regularization_weights[i] = 0.5  # 减少正则化
                elif 'θ' in col:
                    self.feature_weights[i] = 0.5  # 减少θ的权重
                    self.regularization_weights[i] = 4.0  # 增强正则化
                elif 'V' in col:
                    self.feature_weights[i] = 1.5
                    self.regularization_weights[i] = 0.8
        
        # 转换为TensorFlow常量
        self.feature_weights_tf = tf.constant(self.feature_weights.reshape(1, -1), dtype=tf.float32)
        self.regularization_weights_tf = tf.constant(self.regularization_weights.reshape(1, -1), dtype=tf.float32)
        
        # 设置种子
        tf.random.set_seed(seed)
        
        # 使用确定性初始化器
        self.initializer = initializers.GlorotUniform(seed=seed)
        
        # 构建模型
        self.encoder = self._build_encoder()
        self.decoder = self._build_decoder()
        
        # 优化器
        self.optimizer = optimizers.Adam(learning_rate=learning_rate)
        
        self.best_weights = None
        self.best_val_loss = float('inf')
        
        # 固定的随机数生成器
        self.random_generator = tf.random.Generator.from_seed(seed)
    
    def _build_encoder(self):
        """构建编码器"""
        inputs = layers.Input(shape=(self.data_dim,))
        
        x = layers.Dense(128, kernel_initializer=self.initializer,
                        kernel_regularizer=regularizers.l2(0.0003))(inputs)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.Dropout(0.1, seed=self.seed)(x)
        
        x = layers.Dense(64, kernel_initializer=self.initializer,
                        kernel_regularizer=regularizers.l2(0.0003))(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.Dropout(0.08, seed=self.seed+1)(x)
        
        x = layers.Dense(32, kernel_initializer=self.initializer,
                        kernel_regularizer=regularizers.l2(0.0002))(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        
        z_mean = layers.Dense(self.latent_dim, kernel_initializer=self.initializer)(x)
        z_log_var = layers.Dense(self.latent_dim, kernel_initializer=self.initializer)(x)
        
        # 限制log_var的范围
        z_log_var = layers.Lambda(lambda x: tf.clip_by_value(x, -3, 3))(z_log_var)
        
        return keras.Model(inputs, [z_mean, z_log_var])
    
    def _build_decoder(self):
        """构建解码器"""
        inputs = layers.Input(shape=(self.latent_dim,))
        
        x = layers.Dense(32, kernel_initializer=self.initializer,
                        kernel_regularizer=regularizers.l2(0.0002))(inputs)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        
        x = layers.Dense(64, kernel_initializer=self.initializer,
                        kernel_regularizer=regularizers.l2(0.0003))(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.Dropout(0.08, seed=self.seed+2)(x)
        
        x = layers.Dense(128, kernel_initializer=self.initializer,
                        kernel_regularizer=regularizers.l2(0.0003))(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.Dropout(0.1, seed=self.seed+3)(x)
        
        outputs = layers.Dense(self.data_dim, activation='sigmoid', 
                              kernel_initializer=self.initializer)(x)
        
        return keras.Model(inputs, outputs)
    
    def _sample(self, z_mean, z_log_var, training=True):
        """重参数化技巧"""
        if training:
            batch = tf.shape(z_mean)[0]
            dim = tf.shape(z_mean)[1]
            epsilon = self.random_generator.normal(shape=(batch, dim))
            return z_mean + tf.exp(0.5 * z_log_var) * epsilon * 0.06
        else:
            return z_mean
    
    @tf.function
    def train_step(self, x_batch, mask_batch, kl_weight=0.0003):
        """训练步骤"""
        with tf.GradientTape() as tape:
            # 编码
            z_mean, z_log_var = self.encoder(x_batch, training=True)
            z = self._sample(z_mean, z_log_var, training=True)
            
            # 解码
            reconstructed = self.decoder(z, training=True)
            
            # 计算重建损失
            mse = tf.square(x_batch - reconstructed) * mask_batch
            
            # 应用特征权重
            batch_size = tf.shape(x_batch)[0]
            weights = tf.tile(self.feature_weights_tf, [batch_size, 1])
            
            weighted_mse = mse * weights
            weighted_mask = mask_batch * weights
            
            reconstruction_loss = tf.reduce_sum(weighted_mse) / (tf.reduce_sum(weighted_mask) + 1e-8)
            
            # KL损失
            kl_loss = -0.5 * tf.reduce_mean(
                1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var)
            )
            
            # 总损失
            total_loss = reconstruction_loss + kl_weight * kl_loss
        
        # 梯度更新
        variables = self.encoder.trainable_variables + self.decoder.trainable_variables
        gradients = tape.gradient(total_loss, variables)
        gradients = [tf.clip_by_norm(g, 1.0) for g in gradients]
        self.optimizer.apply_gradients(zip(gradients, variables))
        
        return total_loss
    
    def fit(self, data, mask, validation_split=0.2, epochs=900, batch_size=64):
        """训练模型"""
        # 数据分割
        n_samples = len(data)
        indices = np.arange(n_samples)
        local_rng = np.random.RandomState(self.seed)
        local_rng.shuffle(indices)
        
        split_point = int(n_samples * (1 - validation_split))
        train_indices = indices[:split_point]
        val_indices = indices[split_point:]
        
        x_train = data[train_indices]
        x_val = data[val_indices]
        mask_train = mask[train_indices]
        mask_val = mask[val_indices]
        
        print(f"训练集: {len(x_train)} 样本")
        print(f"验证集: {len(x_val)} 样本")
        
        # 早停设置
        patience = 70
        patience_counter = 0
        
        # 学习率调度
        lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
            self.learning_rate,
            decay_steps=len(x_train) // batch_size * epochs,
            alpha=0.03
        )
        self.optimizer.learning_rate = lr_schedule
        
        for epoch in range(epochs):
            # 动态调整KL权重
            kl_weight = min(0.0008, 0.00003 * (epoch / 100))
            
            # 训练
            local_rng = np.random.RandomState(self.seed + epoch)
            train_indices_shuffled = local_rng.permutation(len(x_train))
            train_loss = 0
            n_batches = 0
            
            for i in range(0, len(x_train), batch_size):
                batch_indices = train_indices_shuffled[i:i+batch_size]
                loss = self.train_step(
                    x_train[batch_indices],
                    mask_train[batch_indices],
                    kl_weight
                )
                train_loss += loss
                n_batches += 1
            
            train_loss /= n_batches
            
            # 验证
            if epoch % 10 == 0:
                val_loss = 0
                n_val_batches = 0
                
                for i in range(0, len(x_val), batch_size):
                    z_mean, z_log_var = self.encoder(x_val[i:i+batch_size], training=False)
                    z = z_mean
                    reconstructed = self.decoder(z, training=False)
                    
                    mse = tf.square(x_val[i:i+batch_size] - reconstructed) * mask_val[i:i+batch_size]
                    val_loss += tf.reduce_sum(mse) / tf.reduce_sum(mask_val[i:i+batch_size])
                    n_val_batches += 1
                
                val_loss /= n_val_batches
                
                if val_loss < self.best_val_loss:
                    self.best_val_loss = val_loss
                    self.best_weights = {
                        'encoder': self.encoder.get_weights(),
                        'decoder': self.decoder.get_weights()
                    }
                    patience_counter = 0
                else:
                    patience_counter += 10
                
                if epoch % 50 == 0:
                    print(f"Epoch {epoch}: Train Loss={train_loss:.6f}, Val Loss={val_loss:.6f}")
                
                if patience_counter >= patience:
                    print(f"早停于第 {epoch} 轮")
                    break
        
        # 恢复最佳权重
        if self.best_weights:
            self.encoder.set_weights(self.best_weights['encoder'])
            self.decoder.set_weights(self.best_weights['decoder'])
    
    def impute(self, data, mask):
        """填补缺失值"""
        imputed = np.zeros_like(data)
        batch_size = 128
        
        for i in range(0, len(data), batch_size):
            batch_data = data[i:i+batch_size]
            batch_mask = mask[i:i+batch_size]
            
            z_mean, _ = self.encoder(batch_data, training=False)
            pred = self.decoder(z_mean, training=False).numpy()
            
            imputed[i:i+batch_size] = batch_data * batch_mask + pred * (1 - batch_mask)
        
        return imputed

# 5. 优化的特征专属深度学习模型
class OptimizedFeatureModels:
    def __init__(self, seed=42, column_names=None):
        self.seed = seed
        self.column_names = column_names
        self.models = {}
        self.scalers = {}
    
    def _build_feature_model(self, input_dim, feature_type='normal'):
        """为单个特征构建优化的深度学习模型"""
        tf.random.set_seed(self.seed)
        
        initializer = initializers.GlorotUniform(seed=self.seed)
        
        if feature_type == 'h_feature':
            # H(m)保持当前结构（已经良好）
            model = keras.Sequential([
                layers.Dense(64, activation='relu', input_shape=(input_dim,),
                            kernel_initializer=initializer,
                            kernel_regularizer=regularizers.l2(0.0005)),
                layers.BatchNormalization(),
                layers.Dropout(0.05, seed=self.seed),
                layers.Dense(32, activation='relu', kernel_initializer=initializer,
                            kernel_regularizer=regularizers.l2(0.0005)),
                layers.BatchNormalization(),
                layers.Dense(16, activation='relu', kernel_initializer=initializer),
                layers.Dense(8, activation='relu', kernel_initializer=initializer),
                layers.Dense(1, kernel_initializer=initializer)
            ])
        elif feature_type == 'as_feature':
            # As使用更深的网络，减少正则化
            model = keras.Sequential([
                layers.Dense(96, activation='relu', input_shape=(input_dim,),
                            kernel_initializer=initializer,
                            kernel_regularizer=regularizers.l2(0.0003)),
                layers.BatchNormalization(),
                layers.Dropout(0.08, seed=self.seed),
                layers.Dense(48, activation='relu', kernel_initializer=initializer,
                            kernel_regularizer=regularizers.l2(0.0003)),
                layers.BatchNormalization(),
                layers.Dense(24, activation='relu', kernel_initializer=initializer),
                layers.Dense(12, activation='relu', kernel_initializer=initializer),
                layers.Dense(1, kernel_initializer=initializer)
            ])
        elif feature_type == 'theta_overfitted':
            # θ使用极简网络（强正则化）
            model = keras.Sequential([
                layers.Dense(12, activation='relu', input_shape=(input_dim,),
                            kernel_initializer=initializer,
                            kernel_regularizer=regularizers.l2(0.03)),
                layers.Dropout(0.5, seed=self.seed),
                layers.Dense(6, activation='relu', kernel_initializer=initializer,
                            kernel_regularizer=regularizers.l2(0.03)),
                layers.Dense(1, kernel_initializer=initializer,
                            kernel_regularizer=regularizers.l2(0.02))
            ])
        else:
            # 其他特征使用标准网络
            model = keras.Sequential([
                layers.Dense(32, activation='relu', input_shape=(input_dim,),
                            kernel_initializer=initializer,
                            kernel_regularizer=regularizers.l2(0.005)),
                layers.Dropout(0.2, seed=self.seed),
                layers.Dense(16, activation='relu', kernel_initializer=initializer),
                layers.Dropout(0.1, seed=self.seed+1),
                layers.Dense(8, activation='relu', kernel_initializer=initializer),
                layers.Dense(1, kernel_initializer=initializer)
            ])
        
        # 根据特征类型调整学习率
        if feature_type == 'h_feature':
            lr = 0.003
        elif feature_type == 'as_feature':
            lr = 0.004  # 增加As的学习率
        elif feature_type == 'theta_overfitted':
            lr = 0.0002  # 大幅减少θ的学习率
        else:
            lr = 0.001
        
        model.compile(optimizer=optimizers.Adam(learning_rate=lr), loss='mse')
        return model
    
    def fit(self, data, mask):
        """为每个特征训练专门的深度模型"""
        print("训练优化的特征专属深度模型...")
        
        for col_idx in range(data.shape[1]):
            if self.column_names is not None:
                col_name = self.column_names[col_idx]
            else:
                col_name = f"feature_{col_idx}"
            
            obs_mask = mask[:, col_idx] == 1
            
            if np.sum(obs_mask) < 100:
                continue
            
            # 准备训练数据
            X_train = np.delete(data[obs_mask], col_idx, axis=1)
            y_train = data[obs_mask, col_idx]
            
            # 标准化
            scaler_x = StandardScaler()
            scaler_y = StandardScaler()
            X_train_scaled = scaler_x.fit_transform(X_train)
            y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
            
            # 设置种子
            tf.random.set_seed(self.seed + col_idx)
            np.random.seed(self.seed + col_idx)
            
            # 判断特征类型
            if 'H' in col_name:
                feature_type = 'h_feature'
                patience = 40
                epochs = 350
            elif 'As' in col_name:
                feature_type = 'as_feature'
                patience = 50
                epochs = 400  # 增加训练轮数
            elif 'θ' in col_name:
                feature_type = 'theta_overfitted'
                patience = 8  # 减少耐心值
                epochs = 60  # 减少训练轮数
            else:
                feature_type = 'normal'
                patience = 20
                epochs = 150
            
            # 构建和训练模型
            model = self._build_feature_model(X_train.shape[1], feature_type)
            
            early_stop = callbacks.EarlyStopping(monitor='loss', patience=patience, restore_best_weights=True)
            
            model.fit(
                X_train_scaled, y_train_scaled,
                epochs=epochs,
                batch_size=32,
                verbose=0,
                callbacks=[early_stop],
                shuffle=False
            )
            
            self.models[col_idx] = model
            self.scalers[col_idx] = {'x': scaler_x, 'y': scaler_y}
    
    def impute(self, data, mask):
        """填补缺失值"""
        imputed = data.copy()
        
        for col_idx, model in self.models.items():
            miss_mask = mask[:, col_idx] == 0
            if np.sum(miss_mask) == 0:
                continue
            
            X_miss = np.delete(imputed[miss_mask], col_idx, axis=1)
            X_miss_scaled = self.scalers[col_idx]['x'].transform(X_miss)
            pred_scaled = model.predict(X_miss_scaled, verbose=0).flatten()
            pred = self.scalers[col_idx]['y'].inverse_transform(pred_scaled.reshape(-1, 1)).flatten()
            
            imputed[miss_mask, col_idx] = pred
        
        return imputed

# 6. 优化的集成模型（移除HistGradientBoosting）
class OptimizedEnsemble:
    def __init__(self, data_dim, seed=42, column_names=None):
        self.data_dim = data_dim
        self.seed = seed
        self.column_names = column_names
        self.models = []
        self.weights = []
        self.feature_weights = {}
    
    def fit(self, data, mask, column_names):
        """训练优化的集成模型（不含HistGradientBoosting）"""
        print("\n训练优化的集成模型（不含HistGradientBoosting）...")
        
        # 1. KNN
        print("1. KNN模型...")
        knn = KNNImputer(n_neighbors=20, weights='distance')
        knn.fit(data)
        self.models.append(('knn', knn))
        
        # 2. GradientBoosting IterativeImputer
        print("2. GradientBoosting IterativeImputer...")
        gb_mice = IterativeImputer(
            estimator=GradientBoostingRegressor(
                n_estimators=200,  # 增加以弥补移除HGB的影响
                max_depth=7,
                learning_rate=0.1,
                subsample=0.85,
                random_state=self.seed,
                validation_fraction=0.1,
                n_iter_no_change=5
            ),
            max_iter=35,  # 略微增加迭代次数
            random_state=self.seed,
            sample_posterior=False
        )
        gb_mice.fit(data)
        self.models.append(('gb_mice', gb_mice))
        
        # 3. RandomForest IterativeImputer
        print("3. RandomForest IterativeImputer...")
        rf_mice = IterativeImputer(
            estimator=RandomForestRegressor(
                n_estimators=150,  # 增加树的数量以弥补HGB的缺失
                max_depth=12,  # 略微增加深度
                min_samples_split=3,
                min_samples_leaf=2,
                max_features='sqrt',
                random_state=self.seed
            ),
            max_iter=30,  # 略微增加迭代次数
            random_state=self.seed,
            sample_posterior=False
        )
        rf_mice.fit(data)
        self.models.append(('rf_mice', rf_mice))
        
        # 4. ExtraTrees IterativeImputer
        print("4. ExtraTrees IterativeImputer...")
        et_mice = IterativeImputer(
            estimator=ExtraTreesRegressor(
                n_estimators=120,  # 略微增加树的数量
                max_depth=11,
                min_samples_split=3,
                max_features='sqrt',
                random_state=self.seed
            ),
            max_iter=30,  # 略微增加迭代次数
            random_state=self.seed,
            sample_posterior=False
        )
        et_mice.fit(data)
        self.models.append(('et_mice', et_mice))
        
        # 5. 特征专属深度模型
        print("5. 特征专属深度模型...")
        if hasattr(column_names, 'tolist'):
            column_names_list = column_names.tolist()
        else:
            column_names_list = list(column_names) if column_names is not None else None
        
        fs_model = OptimizedFeatureModels(seed=self.seed, column_names=column_names_list)
        fs_model.fit(data, mask)
        self.models.append(('feature_deep', fs_model))
        
        # 6. 优化的VAE
        print("6. 优化的VAE模型...")
        vae = OptimizedVAE(data_dim=self.data_dim, latent_dim=14,
                          learning_rate=0.0004, seed=self.seed, 
                          column_names=column_names_list)
        
        # 初始填补
        mean_imputer = SimpleImputer(strategy='mean')
        mean_imputer.fit(data)
        initial_imputed = mean_imputer.transform(data)
        data_filled = np.where(mask == 1, data, initial_imputed)
        
        vae.fit(data_filled, mask, epochs=900, batch_size=64)
        self.models.append(('vae', vae))
        
        # 设置优化的特征权重（重新分配移除HGB后的权重）
        for i, col_name in enumerate(column_names):
            if 'V' in col_name:
                # V(m³)权重 - 将原HGB权重分配给其他模型
                self.feature_weights[i] = {
                    'knn': 0.12,  # +0.02
                    'gb_mice': 0.30,  # +0.10
                    'rf_mice': 0.25,  # +0.10
                    'et_mice': 0.13,  # +0.03
                    'feature_deep': 0.15,
                    'vae': 0.05
                }
            elif 'H' in col_name:
                # H(m)权重 - 将原HGB权重主要分配给GB
                self.feature_weights[i] = {
                    'knn': 0.10,
                    'gb_mice': 0.50,  # +0.20（吸收大部分HGB权重）
                    'rf_mice': 0.15,  # +0.05
                    'et_mice': 0.10,  # +0.05
                    'feature_deep': 0.10,
                    'vae': 0.05
                }
            elif 'As' in col_name:
                # As(m²)权重 - 将HGB权重分配给GB和RF
                self.feature_weights[i] = {
                    'knn': 0.08,
                    'gb_mice': 0.40,  # +0.15
                    'rf_mice': 0.35,  # +0.15
                    'et_mice': 0.10,
                    'feature_deep': 0.05,
                    'vae': 0.02
                }
            elif 'θ' in col_name:
                # θ(°)权重 - 保持简单模型为主
                self.feature_weights[i] = {
                    'knn': 0.45,  # +0.05
                    'gb_mice': 0.35,  # +0.05
                    'rf_mice': 0.08,
                    'et_mice': 0.05,
                    'feature_deep': 0.04,
                    'vae': 0.03
                }
            else:
                # Wsm(m)权重 - 均衡分配HGB权重
                self.feature_weights[i] = {
                    'knn': 0.18,  # +0.03
                    'gb_mice': 0.32,  # +0.12
                    'rf_mice': 0.25,  # +0.10
                    'et_mice': 0.10,
                    'feature_deep': 0.10,
                    'vae': 0.05
                }
        
        print("优化的集成模型训练完成（共6个子模型）")
    
    def impute(self, data, mask):
        """集成预测"""
        n_samples, n_features = data.shape
        imputed = np.zeros_like(data)
        
        # 获取所有模型的预测
        all_predictions = {}
        for name, model in self.models:
            if name == 'vae':
                mean_imputer = SimpleImputer(strategy='mean')
                mean_imputer.fit(data)
                initial_imputed = mean_imputer.transform(data)
                data_filled = np.where(mask == 1, data, initial_imputed)
                pred = model.impute(data_filled, mask)
            elif name == 'feature_deep':
                pred = model.impute(data, mask)
            else:
                pred = model.transform(data)
            all_predictions[name] = pred
        
        # 对每个特征使用权重进行加权平均
        for feature_idx in range(n_features):
            if feature_idx in self.feature_weights:
                weights = self.feature_weights[feature_idx]
                feature_pred = np.zeros(n_samples)
                
                for model_name, weight in weights.items():
                    if model_name in all_predictions:
                        feature_pred += all_predictions[model_name][:, feature_idx] * weight
                
                imputed[:, feature_idx] = feature_pred
            else:
                # 使用均值
                feature_pred = np.zeros(n_samples)
                n_models = len(all_predictions)
                for pred in all_predictions.values():
                    feature_pred += pred[:, feature_idx] / n_models
                imputed[:, feature_idx] = feature_pred
        
        # 确保观测值不变
        imputed = data * mask + imputed * (1 - mask)
        
        return imputed

# 7. 评估函数（保持不变）
def deterministic_evaluate(model, data_array, mask, column_names, 
                          data_raw, scalers, transform_info):
    """使用确定性交叉验证评估模型"""
    print("\n=== 使用原始尺度评估模型性能（确定性） ===")
    
    results = []
    all_observations = []
    
    original_index = data_raw.index
    
    for col_idx, col_name in enumerate(column_names):
        obs_mask = mask[:, col_idx] == 1
        obs_indices = np.where(obs_mask)[0]
        
        if len(obs_indices) < 100:
            continue
        
        # 使用确定性的5折交叉验证
        kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
        
        all_true = []
        all_pred = []
        all_indices = []
        
        for fold_idx, (train_idx, test_idx) in enumerate(kf.split(obs_indices)):
            test_indices = obs_indices[test_idx]
            
            # 创建测试掩码
            test_mask = mask.copy()
            test_mask[test_indices, col_idx] = 0
            
            # 填补（标准化尺度）
            imputed_normalized = model.impute(data_array, test_mask)
            
            # 转换回原始尺度
            imputed_original = inverse_transform_array(
                imputed_normalized, scalers, transform_info, 
                column_names, original_index
            )
            
            # 获取原始尺度的真实值
            true_values_original = data_raw.iloc[test_indices][col_name].values
            pred_values_original = imputed_original[col_name].iloc[test_indices].values
            
            all_true.extend(true_values_original)
            all_pred.extend(pred_values_original)
            all_indices.extend(test_indices)
            
            # 记录每个预测
            for idx, true_val, pred_val in zip(test_indices, true_values_original, pred_values_original):
                all_observations.append({
                    '行索引': idx,
                    '特征': col_name,
                    '折数': fold_idx,
                    '原始观测值': true_val,
                    '预测值': pred_val,
                    '绝对误差': abs(true_val - pred_val)
                })
        
        all_true = np.array(all_true)
        all_pred = np.array(all_pred)
        
        # 使用原始尺度计算指标
        mae = mean_absolute_error(all_true, all_pred)
        rmse = np.sqrt(mean_squared_error(all_true, all_pred))
        
        # 标准R²计算
        ss_res = np.sum((all_true - all_pred) ** 2)
        ss_tot = np.sum((all_true - np.mean(all_true)) ** 2)
        r2 = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
        
        # 相关系数
        if np.std(all_true) > 0 and np.std(all_pred) > 0:
            correlation = np.corrcoef(all_true, all_pred)[0, 1]
        else:
            correlation = 0
        
        results.append({
            '特征': col_name,
            'R²(原始尺度)': r2,
            '相关系数': correlation,
            'MAE': mae,
            'RMSE': rmse,
            '样本数': len(all_true)
        })
        
        print(f"{col_name}: R²={r2:.4f}, 相关系数={correlation:.4f}, MAE={mae:.4f}")
    
    return pd.DataFrame(results), pd.DataFrame(all_observations)

# 8. 主函数
def main():
    """主函数"""
    print("=" * 80)
    print("优化的VAE缺失值填补模型 - 移除HistGradientBoosting版本")
    print("=" * 80)
    
    try:
        # 重新设置确定性模式
        set_deterministic_mode()
        
        # 1. 加载数据
        print("\n1. 加载数据...")
        data_raw, original_data = load_and_preprocess_data()
        
        # 保存原始数据用于评估
        data_raw_copy = data_raw.copy()
        
        # 2. 异常值处理
        print("\n2. 异常值处理...")
        data_clean, transform_info = optimized_outlier_handling(data_raw)
        
        # 3. 标准化
        print("\n3. 数据标准化...")
        data_normalized, scalers = optimized_normalization(data_clean)
        
        # 创建掩码
        mask = (~data_normalized.isnull()).astype(np.float32).values
        data_array = data_normalized.fillna(0.5).values.astype(np.float32)
        
        print(f"\n数据形状: {data_array.shape}")
        print(f"缺失率: {(1 - mask.mean())*100:.2f}%")
        
        # 4. 训练优化的集成模型
        print("\n4. 训练优化的集成模型...")
        model = OptimizedEnsemble(data_dim=data_array.shape[1], seed=SEED, 
                                 column_names=data_normalized.columns)
        model.fit(data_array, mask, data_normalized.columns)
        
        # 5. 评估
        print("\n5. 评估模型...")
        eval_results, observations_df = deterministic_evaluate(
            model, data_array, mask, data_normalized.columns, 
            data_raw_copy, scalers, transform_info
        )
        
        # 6. 最终填补
        print("\n6. 生成最终填补数据...")
        final_imputed_normalized = model.impute(data_array, mask)
        
        # 转换回原始尺度
        final_imputed_original = inverse_transform_array(
            final_imputed_normalized, scalers, transform_info, 
            data_normalized.columns, data_raw_copy.index
        )
        
        # 创建完整的填补结果
        final_result = data_raw_copy.copy()
        for col in final_result.columns:
            missing_mask = data_raw_copy[col].isna()
            final_result.loc[missing_mask, col] = final_imputed_original.loc[missing_mask, col]
        
        # 7. 输出结果
        print("\n" + "=" * 80)
        print("模型性能总结")
        print("=" * 80)
        
        for _, row in eval_results.iterrows():
            print(f"\n{row['特征']}:")
            print(f"  R²(原始尺度): {row['R²(原始尺度)']:.4f}")
            print(f"  相关系数: {row['相关系数']:.4f}")
            print(f"  MAE: {row['MAE']:.4f}")
            print(f"  RMSE: {row['RMSE']:.4f}")
            print(f"  评估样本数: {row['样本数']}")
        
        print(f"\n平均R²(原始尺度): {eval_results['R²(原始尺度)'].mean():.4f}")
        print(f"最低R²(原始尺度): {eval_results['R²(原始尺度)'].min():.4f}")
        
        # 判断是否达到要求
        min_r2 = eval_results['R²(原始尺度)'].min()
        if min_r2 >= 0.8:
            print("\n✅ 所有特征的R²都达到0.8以上！")
            print("✅ 模型平衡良好，无明显过拟合或欠拟合！")
        elif min_r2 >= 0.75:
            print(f"\n⚠️ 最低R²为 {min_r2:.4f}，性能良好")
        else:
            print(f"\n⚠️ 最低R²为 {min_r2:.4f}，需要进一步优化")
        
        # 检查过拟合
        high_r2_features = eval_results[eval_results['R²(原始尺度)'] > 0.94]['特征'].tolist()
        if high_r2_features:
            print(f"\n⚠️ 可能存在轻微过拟合的特征: {', '.join(high_r2_features)}")
            print("   已通过增强正则化进行处理")
        
        # 检查欠拟合
        low_r2_features = eval_results[eval_results['R²(原始尺度)'] < 0.75]['特征'].tolist()
        if low_r2_features:
            print(f"\n⚠️ 可能存在欠拟合的特征: {', '.join(low_r2_features)}")
            print("   需要进一步增强学习能力")
        
        # 保存结果
        observations_df.to_csv("观测值与预测值_无HGB版.csv", index=False, encoding='utf-8-sig')
        final_result.to_csv("完整填补结果_无HGB版.csv", index=False, encoding='utf-8-sig')
        eval_results.to_csv("评估结果汇总_无HGB版.csv", index=False, encoding='utf-8-sig')
        
        print("\n文件已保存：")
        print("1. 观测值与预测值_无HGB版.csv")
        print("2. 完整填补结果_无HGB版.csv")
        print("3. 评估结果汇总_无HGB版.csv")
        
        return final_result, eval_results, observations_df
        
    except Exception as e:
        print(f"\n错误: {e}")
        import traceback
        traceback.print_exc()
        return None, None, None

if __name__ == "__main__":
    print("\n程序开始运行（移除HistGradientBoosting版本）...")
    print("主要改进：")
    print("1. 移除HistGradientBoosting IterativeImputer模型")
    print("2. 增强GradientBoosting和RandomForest的参数以弥补影响")
    print("3. 重新分配各特征的模型权重")
    print("4. 集成模型从7个减少到6个子模型")
    print("5. 保持其他优化策略不变")
    
    start_time = time.time()
    
    result, evaluation, observations = main()
    
    print(f"\n✅ 程序运行完成！")
    print(f"总用时: {(time.time() - start_time)/60:.2f} 分钟")
    
    gc.collect()

TensorFlow版本: 2.21.0
已启用完全确定性模式
已禁用GPU，使用CPU确保完全可重复性


程序开始运行（移除HistGradientBoosting版本）...
主要改进：
1. 移除HistGradientBoosting IterativeImputer模型
2. 增强GradientBoosting和RandomForest的参数以弥补影响
3. 重新分配各特征的模型权重
4. 集成模型从7个减少到6个子模型
5. 保持其他优化策略不变
优化的VAE缺失值填补模型 - 移除HistGradientBoosting版本
已启用完全确定性模式

1. 加载数据...

错误: [Errno 2] No such file or directory: './Rainfall.csv'

✅ 程序运行完成！
总用时: 0.00 分钟


Traceback (most recent call last):
  File "C:\Users\LENOVO\AppData\Local\Temp\ipykernel_10224\3604359494.py", line 1129, in main
    data_raw, original_data = load_and_preprocess_data()
                              ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\LENOVO\AppData\Local\Temp\ipykernel_10224\3604359494.py", line 103, in load_and_preprocess_data
    data = pd.read_csv("./Rainfall.csv", low_memory=True)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Anaconda\Anaconda3\envs\tensorflow\Lib\site-packages\pandas\io\parsers\readers.py", line 873, in read_csv
    return _read(filepath_or_buffer, kwds)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Anaconda\Anaconda3\envs\tensorflow\Lib\site-packages\pandas\io\parsers\readers.py", line 300, in _read
    parser = TextFileReader(filepath_or_buffer, **kwds)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Anaconda\Anaconda3\envs\tensorflow\Lib\site-packages\pandas\io\parsers\readers.py", line

In [2]:
import pandas as pd

def add_L_column_to_results():
    """
    将L(m)列添加到已生成的填补结果文件中
    """
    # 1. 读取原始数据以获取L(m)列
    original_data = pd.read_csv("./Rainfall.csv", low_memory=True)
    
    # 2. 检查L(m)列是否存在
    if 'L(m)' not in original_data.columns:
        print("原始数据中未找到L(m)列")
        return
    
    # 3. 提取L(m)列
    L_column = original_data['L(m)'].copy()
    print(f"成功提取L(m)列，共{len(L_column)}行")
    
    # 4. 读取填补结果
    try:
        imputed_results = pd.read_csv("完整填补结果_无HGB版.csv")
        print(f"填补结果形状: {imputed_results.shape}")
    except FileNotFoundError:
        print("未找到填补结果文件，请先运行主程序生成结果")
        return
    
    # 5. 检查行数是否匹配
    if len(imputed_results) != len(L_column):
        print(f"警告：行数不匹配！填补结果有{len(imputed_results)}行，L(m)列有{len(L_column)}行")
        # 如果行数不同，需要对齐（取最小行数）
        min_rows = min(len(imputed_results), len(L_column))
        imputed_results = imputed_results.iloc[:min_rows]
        L_column = L_column.iloc[:min_rows]
    
    # 6. 添加L(m)列到结果中
    imputed_results['L(m)'] = L_column.values
    
    # 7. 保存更新后的结果
    output_filename = "填补结果_Rainfall_包含L列.csv"
    imputed_results.to_csv(output_filename, index=False)
    
    print(f"\n成功添加L(m)列到结果中！")
    print(f"新文件已保存为: {output_filename}")
    print(f"最终数据形状: {imputed_results.shape}")
    print(f"列名: {list(imputed_results.columns)}")
    
    # 8. 显示L(m)列的基本统计信息
    print("\nL(m)列统计信息:")
    print(f"  数据类型: {L_column.dtype}")
    print(f"  唯一值数量: {L_column.nunique()}")
    print(f"  缺失值数量: {L_column.isna().sum()}")
    if pd.api.types.is_numeric_dtype(L_column):
        print(f"  最小值: {L_column.min()}")
        print(f"  最大值: {L_column.max()}")
        print(f"  平均值: {L_column.mean():.2f}")
    else:
        print(f"  前5个值: {L_column.head().tolist()}")
    
    return imputed_results

# 运行函数
if __name__ == "__main__":
    result_with_L = add_L_column_to_results()

成功提取L(m)列，共17486行
填补结果形状: (17486, 5)

成功添加L(m)列到结果中！
新文件已保存为: 填补结果_Rainfall_包含L列.csv
最终数据形状: (17486, 6)
列名: ['As(m²）', 'H(m)', 'V(m³)', 'Wsm(m)', 'θ(°)', 'L(m)']

L(m)列统计信息:
  数据类型: float64
  唯一值数量: 8135
  缺失值数量: 0
  最小值: 0.42
  最大值: 4590.0
  平均值: 62.20
